<a href="https://colab.research.google.com/github/john891212-oss/AIFFEL_quest_eng/blob/main/NLP/NLP05/happy_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

프로젝트 제목

한화 팬들의 “나는 행복합니다”는 정말 행복한가?   
경기 결과 기반 한화 이글스 댓글 감정 분석 모델 비교

In [ ]:
!pip install -U transformers datasets evaluate accelerate peft scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling trans

In [ ]:
import torch, transformers, datasets, sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Transformers: 5.9.0
Datasets: 4.8.5


모델 로딩

In [ ]:
BASE_MODEL = "klue/roberta-base"
COMPARE_MODEL = "beomi/KcELECTRA-base"

num_labels = 5

id2label = {
    0: "기쁨/행복",
    1: "분노/실망",
    2: "체념/해탈",
    3: "중립/정보성"
}

label2id = {v: k for k, v in id2label.items()}

유튜브 댓글 크롤링 라벨링 자동화하여 csv 파일화

In [ ]:
!pip install -q google-api-python-client pandas tqdm emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.3 MB/s eta 0:00:00


api key

In [ ]:
from getpass import getpass
from googleapiclient.discovery import build

YOUTUBE_API_KEY = getpass("YouTube API Key 입력: ")

youtube = build(
    "youtube",
    "v3",
    developerKey=YOUTUBE_API_KEY
)

YouTube API Key 입력: ··········


영상메타 csv 파일 만들기

In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse, parse_qs

video_urls = [
    "https://www.youtube.com/watch?v=pOb2Fx0-Qmw",
    "https://www.youtube.com/watch?v=x86ZLds-CuM",
    "https://www.youtube.com/watch?v=7NJNvlFSRFU",
    "https://www.youtube.com/watch?v=hqVvBuoZy2Y",
    "https://www.youtube.com/watch?v=hvUXqcB4kBU",
    "https://www.youtube.com/watch?v=ccD49bkIG-I",
    "https://www.youtube.com/watch?v=625A0IgGH90",
    "https://www.youtube.com/watch?v=en4KbHRYyGo",
    "https://www.youtube.com/watch?v=n1ZBRnryMLU&t=32s",
    "https://www.youtube.com/watch?v=_NAqs8ZDYC8",
]

def extract_video_id(url_or_id):
    text = str(url_or_id).strip()

    if re.fullmatch(r"[a-zA-Z0-9_-]{11}", text):
        return text

    parsed = urlparse(text)

    if parsed.netloc in ["www.youtube.com", "youtube.com", "m.youtube.com"]:
        if parsed.path == "/watch":
            query = parse_qs(parsed.query)
            return query.get("v", [None])[0]

    if parsed.netloc in ["youtu.be", "www.youtu.be"]:
        return parsed.path.strip("/").split("/")[0]

    return None


video_meta = pd.DataFrame({
    "video_url": video_urls
})

video_meta["video_id"] = video_meta["video_url"].apply(extract_video_id)

video_meta

,video_url,video_id
0,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,pOb2Fx0-Qmw
1,https://www.youtube.com/watch?v=x86ZLds-CuM,x86ZLds-CuM
2,https://www.youtube.com/watch?v=7NJNvlFSRFU,7NJNvlFSRFU
3,https://www.youtube.com/watch?v=hqVvBuoZy2Y,hqVvBuoZy2Y
4,https://www.youtube.com/watch?v=hvUXqcB4kBU,hvUXqcB4kBU
5,https://www.youtube.com/watch?v=ccD49bkIG-I,ccD49bkIG-I
6,https://www.youtube.com/watch?v=625A0IgGH90,625A0IgGH90
7,https://www.youtube.com/watch?v=en4KbHRYyGo,en4KbHRYyGo
8,https://www.youtube.com/watch?v=n1ZBRnryMLU&t=32s,n1ZBRnryMLU
9,https://www.youtube.com/watch?v=_NAqs8ZDYC8,_NAqs8ZDYC8


In [ ]:
def get_video_title(video_id):
    try:
        response = youtube.videos().list(
            part="snippet",
            id=video_id
        ).execute()

        items = response.get("items", [])
        if not items:
            return None

        return items[0]["snippet"].get("title")

    except Exception as e:
        print(f"[TITLE ERROR] {video_id}: {e}")
        return None


video_meta["video_title"] = video_meta["video_id"].apply(get_video_title)

video_meta

,video_url,video_id,video_title
0,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
1,https://www.youtube.com/watch?v=x86ZLds-CuM,x86ZLds-CuM,[한화 vs KT] 5/17 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
2,https://www.youtube.com/watch?v=7NJNvlFSRFU,7NJNvlFSRFU,[한화 vs KT] 5/16 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
3,https://www.youtube.com/watch?v=hqVvBuoZy2Y,hqVvBuoZy2Y,[한화 vs KT] 5/15 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
4,https://www.youtube.com/watch?v=hvUXqcB4kBU,hvUXqcB4kBU,[한화 vs 키움] 5/14 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
5,https://www.youtube.com/watch?v=ccD49bkIG-I,ccD49bkIG-I,[NC vs 한화] 4/24 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
6,https://www.youtube.com/watch?v=625A0IgGH90,625A0IgGH90,[삼성 vs 한화] 4/16 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
7,https://www.youtube.com/watch?v=en4KbHRYyGo,en4KbHRYyGo,[KIA vs 한화] 4/10 경기 I 2026 신한 SOL KBO 리그 I 하이라...
8,https://www.youtube.com/watch?v=n1ZBRnryMLU&t=32s,n1ZBRnryMLU,[NC vs 한화] 4/26 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
9,https://www.youtube.com/watch?v=_NAqs8ZDYC8,_NAqs8ZDYC8,[한화 vs KIA] 5/7 경기 I 2026 신한 SOL KBO 리그 I 하이라이...


In [ ]:
video_meta["game_date"] = [
    "2026-05-19",
    "2026-05-17",
    "2026-05-16",
    "2026-05-15",
    "2026-05-14",
    "2026-04-24",
    "2026-04-16",
    "2026-04-10",
    "2026-04-26",
    "2026-05-07"
]

video_meta["opponent"] = [
    "롯데",
    "KT",
    "KT",
    "KT",
    "키움",
    "NC",
    "삼성",
    "KIA",
    "NC",
    "KIA"
]

video_meta["result"] = [
    "lose",
    "lose",
    "win",
    "win",
    "win",
    "lose",
    "lose",
    "lose",
    "lose",
    "win"
]

video_meta["score"] = [
    "6:4",
    "7:8",
    "10:5",
    "5:3",
    "10:1",
    "7:3",
    "6:1",
    "6:5",
    "5:3",
    "11:8"
]

video_meta["source"] = "youtube"

video_meta

,video_url,video_id,video_title,game_date,opponent,result,score,source
0,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-19,롯데,lose,6:4,youtube
1,https://www.youtube.com/watch?v=x86ZLds-CuM,x86ZLds-CuM,[한화 vs KT] 5/17 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-17,KT,lose,7:8,youtube
2,https://www.youtube.com/watch?v=7NJNvlFSRFU,7NJNvlFSRFU,[한화 vs KT] 5/16 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-16,KT,win,10:5,youtube
3,https://www.youtube.com/watch?v=hqVvBuoZy2Y,hqVvBuoZy2Y,[한화 vs KT] 5/15 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-15,KT,win,5:3,youtube
4,https://www.youtube.com/watch?v=hvUXqcB4kBU,hvUXqcB4kBU,[한화 vs 키움] 5/14 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-14,키움,win,10:1,youtube
5,https://www.youtube.com/watch?v=ccD49bkIG-I,ccD49bkIG-I,[NC vs 한화] 4/24 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-04-24,NC,lose,7:3,youtube
6,https://www.youtube.com/watch?v=625A0IgGH90,625A0IgGH90,[삼성 vs 한화] 4/16 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-04-16,삼성,lose,6:1,youtube
7,https://www.youtube.com/watch?v=en4KbHRYyGo,en4KbHRYyGo,[KIA vs 한화] 4/10 경기 I 2026 신한 SOL KBO 리그 I 하이라...,2026-04-10,KIA,lose,6:5,youtube
8,https://www.youtube.com/watch?v=n1ZBRnryMLU&t=32s,n1ZBRnryMLU,[NC vs 한화] 4/26 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-04-26,NC,lose,5:3,youtube
9,https://www.youtube.com/watch?v=_NAqs8ZDYC8,_NAqs8ZDYC8,[한화 vs KIA] 5/7 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-07,KIA,win,11:8,youtube


댓글 수집 함수

In [ ]:
import time
from tqdm import tqdm

def get_youtube_comments(
    video_id,
    max_comments=100,
    order="relevance",
    sleep_sec=0.1
):
    comments = []
    next_page_token = None
    error_message = None

    while len(comments) < max_comments:
        try:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat="plainText",
                order=order
            ).execute()

        except Exception as e:
            error_message = str(e)
            print(f"[COMMENT ERROR] video_id={video_id}")
            print(error_message[:500])
            break

        for item in response.get("items", []):
            top_comment = item["snippet"]["topLevelComment"]
            snippet = top_comment["snippet"]

            comments.append({
                "comment_id": top_comment.get("id"),
                "video_id": video_id,
                "comment_text": snippet.get("textOriginal", snippet.get("textDisplay", "")),
                "author": snippet.get("authorDisplayName", ""),
                "like_count": snippet.get("likeCount", 0),
                "published_at": snippet.get("publishedAt", ""),
                "updated_at": snippet.get("updatedAt", ""),
                "total_reply_count": item["snippet"].get("totalReplyCount", 0),
                "collection_error": None,
            })

            if len(comments) >= max_comments:
                break

        next_page_token = response.get("nextPageToken")

        if not next_page_token:
            break

        time.sleep(sleep_sec)

    if len(comments) == 0 and error_message is not None:
        return pd.DataFrame([{
            "comment_id": None,
            "video_id": video_id,
            "comment_text": None,
            "author": None,
            "like_count": None,
            "published_at": None,
            "updated_at": None,
            "total_reply_count": None,
            "collection_error": error_message,
        }])

    return pd.DataFrame(comments)

여러 영상에서 댓글 수집

In [ ]:
video_meta.to_csv(
    "/content/hanwha_video_meta.csv",
    index=False,
    encoding="utf-8-sig"
)
video_meta = pd.read_csv("/content/hanwha_video_meta.csv")

all_comments = []

for _, row in tqdm(video_meta.iterrows(), total=len(video_meta)):
    video_id = row["video_id"]

    comment_df = get_youtube_comments(
        video_id=video_id,
        max_comments=100,      # 영상당 100개
        order="relevance"      # 인기/관련도 높은 댓글 우선
    )

    if len(comment_df) == 0:
        continue

    comment_df["video_url"] = row.get("video_url")
    comment_df["video_title"] = row.get("video_title")
    comment_df["game_date"] = row.get("game_date")
    comment_df["opponent"] = row.get("opponent")
    comment_df["result"] = row.get("result")
    comment_df["score"] = row.get("score")
    comment_df["source"] = row.get("source")

    all_comments.append(comment_df)

raw_df = pd.concat(all_comments, ignore_index=True)

raw_df.to_csv(
    "/content/hanwha_comments_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print("수집된 댓글 수:", len(raw_df))
raw_df.head()

100%|██████████| 10/10 [00:03<00:00,  2.51it/s]

수집된 댓글 수: 1000


,comment_id,video_id,comment_text,author,like_count,published_at,updated_at,total_reply_count,collection_error,video_url,video_title,game_date,opponent,result,score,source
0,UgzGKHYxq5HkIWFVjcR4AaABAg,pOb2Fx0-Qmw,KBO 리그는 오직 티빙에서 ⚾\r\nhttps://tving.onelink.me/...,@tvingsports,32,2026-05-19T13:33:27Z,2026-05-19T13:33:27Z,0,None,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-19,롯데,lose,6:4,youtube
1,Ugw6dS_-wXzRp9m7XlB4AaABAg,pOb2Fx0-Qmw,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,@Wondksjdd,2628,2026-05-19T13:35:18Z,2026-05-19T13:35:18Z,84,None,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-19,롯데,lose,6:4,youtube
2,UgyXfL5CZ32dqcDwDNJ4AaABAg,pOb2Fx0-Qmw,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ\n오늘 골고루 활약해서 더 보기가좋네,@user-mingyuGu,1030,2026-05-19T13:45:43Z,2026-05-19T13:45:43Z,6,None,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-19,롯데,lose,6:4,youtube
3,UgzofTPWigXOc5odOWR4AaABAg,pOb2Fx0-Qmw,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ,@김김-d9k8y,1347,2026-05-19T13:36:51Z,2026-05-19T13:36:51Z,19,None,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-19,롯데,lose,6:4,youtube
4,Ugyra0N4TZK-7nkv8od4AaABAg,pOb2Fx0-Qmw,오늘 경기 전민재 칭찬이 많이 없네\n홈런에 호수비에 활약 좋았는데,@박현우-c7q,914,2026-05-19T13:58:44Z,2026-05-19T13:58:44Z,8,None,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,2026-05-19,롯데,lose,6:4,youtube


In [ ]:
raw_df.groupby(["video_id", "video_title"]).size().reset_index(name="comment_count")

,video_id,video_title,comment_count
0,625A0IgGH90,[삼성 vs 한화] 4/16 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
1,7NJNvlFSRFU,[한화 vs KT] 5/16 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
2,_NAqs8ZDYC8,[한화 vs KIA] 5/7 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
3,ccD49bkIG-I,[NC vs 한화] 4/24 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
4,en4KbHRYyGo,[KIA vs 한화] 4/10 경기 I 2026 신한 SOL KBO 리그 I 하이라...,100
5,hqVvBuoZy2Y,[한화 vs KT] 5/15 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
6,hvUXqcB4kBU,[한화 vs 키움] 5/14 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
7,n1ZBRnryMLU,[NC vs 한화] 4/26 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
8,pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100
9,x86ZLds-CuM,[한화 vs KT] 5/17 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,100


댓글 정제

In [ ]:
import emoji

def clean_comment(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\n", " ")

    # URL 제거
    text = re.sub(r"http\S+|www\S+", " ", text)

    # 멘션 제거
    text = re.sub(r"@\S+", " ", text)

    # 이모지 제거
    text = emoji.replace_emoji(text, replace=" ")

    # 감정 표현 보존
    text = re.sub(
        r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\sㅋㅎㅠㅜ!?.,~]",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


df = pd.read_csv("/content/hanwha_comments_raw.csv")

# 수집 실패 행 제거
df = df[df["comment_text"].notna()].copy()

df["text"] = df["comment_text"].apply(clean_comment)

# 너무 짧은 댓글 제거
df = df[df["text"].str.len() >= 3]

# 중복 제거
df = df.drop_duplicates(subset=["video_id", "text"])

df = df.reset_index(drop=True)

df.to_csv(
    "/content/hanwha_comments_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("정제 후 댓글 수:", len(df))
df[["text", "video_title"]].head()

정제 후 댓글 수: 998


,text,video_title
0,KBO 리그는 오직 티빙에서,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
1,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
2,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
3,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...
4,오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...


자동 라벨링  
(실제로 사용하지 않고 수동 라벨링을 했음)

In [ ]:
id2label = {
    0: "긍정/행복",
    1: "분노/실망",
    2: "체념/해탈",
    3: "중립/정보성",
}

def auto_label_hanwha_comment_4class(text, result=None):
    text = str(text).strip()

    scores = {
        0: 0.0,  # 긍정/행복
        1: 0.0,  # 분노/실망
        2: 0.0,  # 체념/해탈
        3: 0.0,  # 중립/정보성
    }
    reasons = []

    positive_keywords = [
        "행복", "이겼", "승리", "연승", "나이스", "최고", "고맙",
        "잘하네", "잘했다", "미쳤다", "미쳤네", "갓", "굿", "대박",
        "사랑", "보물", "살아났다", "좋다", "좋네",
        "장타", "타점", "홈런", "호수비", "활약", "멀티",
        "거포", "안아깝다", "이쁘게", "희망", "기대", "가능성",
        "미래", "유망주", "성장", "포텐", "괜찮", "개추"
    ]

    anger_keywords = [
        "빡", "화나", "화난다", "실망", "답없", "답 없다", "못하",
        "최악", "뭐하", "터졌", "불펜", "감독", "교체", "에휴",
        "노답", "짜증", "망", "졌냐", "지냐", "개못",
        "볼넷", "어이없", "한심", "자동", "병살", "헛스윙"
    ]

    resignation_keywords = [
        "한화했다", "또 졌", "또 지", "익숙", "해탈", "그러려니",
        "아무렇지도", "기대도 안", "아쉽", "이게 한화", "한화답다",
        "나는 행복합니다", "행복합니다", "올해도", "언제나",
        "그럼 그렇지", "2군", "불쌍", "걱정"
    ]

    neutral_keywords = [
        "선발", "라인업", "몇시", "스코어", "삼팬", "엘팬", "두팬",
        "킅팬", "순위", "기록", "상대", "중계", "약점",
        "경기", "오늘", "하이라이트", "리그"
    ]

    for kw in positive_keywords:
        if kw in text:
            scores[0] += 1.0
            reasons.append(f"positive:{kw}")

    for kw in anger_keywords:
        if kw in text:
            scores[1] += 1.0
            reasons.append(f"anger:{kw}")

    for kw in resignation_keywords:
        if kw in text:
            scores[2] += 1.0
            reasons.append(f"resign:{kw}")

    for kw in neutral_keywords:
        if kw in text:
            scores[3] += 0.5
            reasons.append(f"neutral:{kw}")

    # 경기 결과는 약하게만 보정
    if result == "win":
        scores[0] += 0.3
    elif result == "lose":
        scores[1] += 0.2
        scores[2] += 0.2

    # "나는 행복합니다" 반어 처리
    if "행복합니다" in text or "나는 행복합니다" in text:
        if result == "lose" or "ㅋㅋ" in text or "ㅠ" in text or "또" in text:
            scores[2] += 2.0
            reasons.append("sarcasm:happiness_meme")
        else:
            scores[0] += 1.0
            reasons.append("positive:happiness_phrase")

    # "그래도/졌지만" 긍정 맥락은 긍정/행복으로 통합
    if ("졌지만" in text or "그래도" in text) and any(
        kw in text for kw in ["희망", "가능성", "잘했다", "괜찮", "문동주", "신인", "성장"]
    ):
        scores[0] += 1.5
        reasons.append("positive:lost_but_positive")

    # 근거 없으면 중립
    if max(scores.values()) == 0:
        label = 3
        confidence = 0.3
        reasons.append("default:neutral")
    else:
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        label = sorted_scores[0][0]
        top = sorted_scores[0][1]
        second = sorted_scores[1][1]

        # 1등과 2등 차이가 거의 없으면 confidence 낮게
        confidence = min(0.95, 0.45 + (top - second) * 0.2)

    return label, id2label[label], confidence, "|".join(reasons)

In [ ]:
label_results = df.apply(
    lambda row: auto_label_hanwha_comment_4class(
        row["text"],
        row.get("result", None)
    ),
    axis=1
)

df["label"] = [x[0] for x in label_results]
df["label_name"] = [x[1] for x in label_results]
df["label_confidence"] = [x[2] for x in label_results]
df["label_reason"] = [x[3] for x in label_results]

df[["text", "label", "label_name", "label_confidence", "label_reason"]].head(30)

,text,label,label_name,label_confidence,label_reason
0,KBO 리그는 오직 티빙에서,3,중립/정보성,0.51,neutral:리그
1,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,0,긍정/행복,0.61,positive:개추
2,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네,0,긍정/행복,0.95,positive:좋다|positive:좋네|positive:호수비|positive:...
3,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ,0,긍정/행복,0.61,positive:개추
4,오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데,0,긍정/행복,0.85,positive:홈런|positive:호수비|positive:활약|neutral:경...
5,한동희 3경기 연속홈런이지만 그게 전부여서 아직 의심중인 롯데팬이면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋ,0,긍정/행복,0.75,positive:홈런|positive:개추|neutral:경기
6,14 19 손성빈 3루에 강속구 송구가 대박임 최근 타격도 잘해서 요즘 젤 잘하는듯,0,긍정/행복,0.61,positive:대박
7,0 23 아니비슬리 프로필 볼때마다 저항없이 웃게되네 ㅈㄴ해맑음,1,분노/실망,0.45,
8,전민재는 오랜고민이였던 유격수 보물이 맞다,0,긍정/행복,0.61,positive:보물
9,유강남을 상대하던 팀들이 이런 기분이었을까,3,중립/정보성,0.51,neutral:상대


In [ ]:
df[df["label_confidence"] < 0.6][
    ["text", "label_name", "label_confidence", "label_reason"]
].head(50)

,text,label_name,label_confidence,label_reason
0,KBO 리그는 오직 티빙에서,중립/정보성,0.51,neutral:리그
7,0 23 아니비슬리 프로필 볼때마다 저항없이 웃게되네 ㅈㄴ해맑음,분노/실망,0.45,
9,유강남을 상대하던 팀들이 이런 기분이었을까,중립/정보성,0.51,neutral:상대
11,장두성 부상아니라 쥐나서 다행입니다,분노/실망,0.45,
13,손성빈은 매경기 성장하네,긍정/행복,0.55,positive:성장|neutral:경기
17,오늘 1호기2호기가 ㄹㅇ 후반에 지렸음 맨날당하던거 오늘 맛보니 진짜 도파민 오짐,중립/정보성,0.51,neutral:오늘
22,"장두성, 손성빈 두 선수 칭찬합니다!!!",분노/실망,0.45,
24,한화전 연패 끊어내서 기쁜 진짜 롯데팬만 개추 ㅋㅋ,분노/실망,0.49,positive:개추|anger:연패
28,한동희 3경기 연속 홈런.. 당한게 많지만 이럴때 일수록 칭찬해주자 분위기 잘 이어가게,긍정/행복,0.55,positive:홈런|neutral:경기
30,롯데 응원가 너무 좋더라..ㅎ 부산갈매기~ 부산갈매기~~..,분노/실망,0.45,


자동 라벨링 결과 클래스 불균형이 심하게 나타났다

In [ ]:
df["label_name"].value_counts()

,count
label_name,
긍정/행복,455
분노/실망,424
중립/정보성,91
체념/해탈,28


In [ ]:
final_cols = [
    "text",
    "label",
    "label_name",
    "label_confidence",
    "label_reason",
    "game_date",
    "opponent",
    "result",
    "score",
    "video_id",
    "video_url",
    "video_title",
    "source",
    "like_count",
    "published_at",
    "total_reply_count"
]

final_df = df[final_cols].copy()

final_df.to_csv(
    "/content/hanwha_sentiment_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("최종 데이터 수:", len(final_df))
final_df.head()

최종 데이터 수: 998


,text,label,label_name,label_confidence,label_reason,game_date,opponent,result,score,video_id,video_url,video_title,source,like_count,published_at,total_reply_count
0,KBO 리그는 오직 티빙에서,3,중립/정보성,0.51,neutral:리그,2026-05-19,롯데,lose,6:4,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,youtube,32,2026-05-19T13:33:27Z,0
1,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,0,긍정/행복,0.61,positive:개추,2026-05-19,롯데,lose,6:4,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,youtube,2628,2026-05-19T13:35:18Z,84
2,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네,0,긍정/행복,0.95,positive:좋다|positive:좋네|positive:호수비|positive:...,2026-05-19,롯데,lose,6:4,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,youtube,1030,2026-05-19T13:45:43Z,6
3,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ,0,긍정/행복,0.61,positive:개추,2026-05-19,롯데,lose,6:4,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,youtube,1347,2026-05-19T13:36:51Z,19
4,오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데,0,긍정/행복,0.85,positive:홈런|positive:호수비|positive:활약|neutral:경...,2026-05-19,롯데,lose,6:4,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,youtube,914,2026-05-19T13:58:44Z,8


In [ ]:
final_df["label_name"].value_counts()

,count
label_name,
긍정/행복,455
분노/실망,424
중립/정보성,91
체념/해탈,28


수동 라벨링용 함수

In [ ]:
import pandas as pd

df = pd.read_csv("/content/hanwha_comments_clean.csv")

# 사람이 직접 입력할 컬럼
df["label"] = ""
df["label_name"] = ""
df["memo"] = ""

labeling_df = df[
    [
        "text",
        "label",
        "label_name",
        "memo",
        "video_title",
        "video_id",
        "video_url",
        "game_date",
        "opponent",
        "result",
        "score",
        "like_count",
        "published_at"
    ]
].copy()

labeling_df.to_csv(
    "/content/hanwha_manual_labeling.csv",
    index=False,
    encoding="utf-8-sig"
)

labeling_df.head()

,text,label,label_name,memo,video_title,video_id,video_url,game_date,opponent,result,score,like_count,published_at
0,KBO 리그는 오직 티빙에서,,,,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,2026-05-19,롯데,lose,6:4,32,2026-05-19T13:33:27Z
1,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,,,,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,2026-05-19,롯데,lose,6:4,2628,2026-05-19T13:35:18Z
2,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네,,,,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,2026-05-19,롯데,lose,6:4,1030,2026-05-19T13:45:43Z
3,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ,,,,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,2026-05-19,롯데,lose,6:4,1347,2026-05-19T13:36:51Z
4,오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데,,,,[롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...,pOb2Fx0-Qmw,https://www.youtube.com/watch?v=pOb2Fx0-Qmw,2026-05-19,롯데,lose,6:4,914,2026-05-19T13:58:44Z


다운받은 파일로 라벨링 수작업으로 하기

In [ ]:
from google.colab import files

files.download("/content/hanwha_manual_labeling.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

파일 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


CSV 파일 => train val test로 나누기

In [ ]:
from datasets import load_dataset

base_path = "/content/drive/MyDrive/Colab Notebooks"

manual_labeled_path = f"{base_path}/hanwha_manual_labeling 1000.csv"

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import load_dataset
# 1. 수동 라벨링 데이터 읽기
df = pd.read_csv(manual_labeled_path)

print("원본 데이터 수:", len(df))
print(df.head())
print(df.columns)

원본 데이터 수: 998
                                            text  label  label_name  memo  \
0                                KBO 리그는 오직 티빙에서      3         NaN   NaN   
1        한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추      3         NaN   NaN   
2  손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네      0         NaN   NaN   
3                     손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ      0         NaN   NaN   
4           오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데      0         NaN   NaN   

                                         video_title     video_id  \
0  [롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...  pOb2Fx0-Qmw   
1  [롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...  pOb2Fx0-Qmw   
2  [롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...  pOb2Fx0-Qmw   
3  [롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...  pOb2Fx0-Qmw   
4  [롯데 vs 한화] 5/19 경기 I 2026 신한 SOL KBO 리그 I 하이라이...  pOb2Fx0-Qmw   

                                     video_url   game_date opponent result  \
0  https://www.youtube.com/wat

In [ ]:
# label 숫자화
df["label"] = pd.to_numeric(df["label"], errors="coerce")

# 라벨 없는 행 제거
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

# 4분류 학습용: 0,1,2,3만 사용 / 9는 제외
df = df[df["label"].isin([0, 1, 2, 3])].copy()

# text 결측 제거
df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str)

# 중복 제거
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

id2label_4 = {
    0: "긍정/행복",
    1: "분노/실망",
    2: "체념/해탈",
    3: "중립/정보성",
}

df["label_name"] = df["label"].map(id2label_4)

print("학습 사용 데이터 수:", len(df))
print(df["label_name"].value_counts())

학습 사용 데이터 수: 992
label_name
긍정/행복     480
분노/실망     278
중립/정보성    132
체념/해탈     102
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

print("\nTrain")
print(train_df["label_name"].value_counts())

print("\nValid")
print(valid_df["label_name"].value_counts())

print("\nTest")
print(test_df["label_name"].value_counts())

train: 694
valid: 149
test: 149

Train
label_name
긍정/행복     336
분노/실망     195
중립/정보성     92
체념/해탈      71
Name: count, dtype: int64

Valid
label_name
긍정/행복     72
분노/실망     41
중립/정보성    20
체념/해탈     16
Name: count, dtype: int64

Test
label_name
긍정/행복     72
분노/실망     42
중립/정보성    20
체념/해탈     15
Name: count, dtype: int64


In [ ]:
train_path = f"{base_path}/hanwha_train_4class_1000.csv"
valid_path = f"{base_path}/hanwha_valid_4class_1000.csv"
test_path = f"{base_path}/hanwha_test_4class_1000.csv"

train_df.to_csv(train_path, index=False, encoding="utf-8-sig")
valid_df.to_csv(valid_path, index=False, encoding="utf-8-sig")
test_df.to_csv(test_path, index=False, encoding="utf-8-sig")

print("저장 완료")
print(train_path)
print(valid_path)
print(test_path)

저장 완료
/content/drive/MyDrive/Colab Notebooks/hanwha_train_4class_1000.csv
/content/drive/MyDrive/Colab Notebooks/hanwha_valid_4class_1000.csv
/content/drive/MyDrive/Colab Notebooks/hanwha_test_4class_1000.csv


KOTE 모델로 댓글 CSV 라벨링 테스트 해보기

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline

model_name = "searle-j/kote_for_easygoing_people"

kote_tokenizer = AutoTokenizer.from_pretrained(model_name)
kote_model = AutoModelForSequenceClassification.from_pretrained(model_name)

kote_pipe = TextClassificationPipeline(
    model=kote_model,
    tokenizer=kote_tokenizer,
    device=0,          # GPU 사용, CPU면 -1
    top_k=5,
    function_to_apply="sigmoid"
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [ ]:
texts = [
    "나는 행복합니다 ㅋㅋㅋㅋ 또 졌네",
    "오늘 한화 진짜 잘했다",
    "불펜 진짜 답없다",
    "오늘 선발 누구임?"
]

for t in texts:
    print(t)
    print(kote_pipe(t))
    print()

나는 행복합니다 ㅋㅋㅋㅋ 또 졌네
[[{'label': '즐거움/신남', 'score': 0.7401877045631409}, {'label': '기쁨', 'score': 0.6973012685775757}, {'label': '행복', 'score': 0.6722671985626221}, {'label': '힘듦/지침', 'score': 0.5869739651679993}, {'label': '깨달음', 'score': 0.4241591691970825}]]

오늘 한화 진짜 잘했다
[[{'label': '기쁨', 'score': 0.9507696032524109}, {'label': '즐거움/신남', 'score': 0.8902125358581543}, {'label': '감동/감탄', 'score': 0.8798384666442871}, {'label': '행복', 'score': 0.855192244052887}, {'label': '고마움', 'score': 0.8458854556083679}]]

불펜 진짜 답없다
[[{'label': '불평/불만', 'score': 0.9519044160842896}, {'label': '짜증', 'score': 0.942493200302124}, {'label': '안타까움/실망', 'score': 0.9217226505279541}, {'label': '화남/분노', 'score': 0.9039307236671448}, {'label': '한심함', 'score': 0.8561615943908691}]]

오늘 선발 누구임?
[[{'label': '기대감', 'score': 0.9324690103530884}, {'label': '신기함/관심', 'score': 0.7661044597625732}, {'label': '없음', 'score': 0.5182512402534485}, {'label': '불안/걱정', 'score': 0.3274783492088318}, {'label': '환영/호의', 'score

반어법이나 밈이 섞인 댓글은 잘 파악하지 못한다.   
4분류 라벨링에 보조적인 품질 점검용으로 사용.

In [ ]:
kote_to_hanwha_4 = {
    # 0 긍정/행복
    "환영/호의": 0,
    "감동/감탄": 0,
    "고마움": 0,
    "존경": 0,
    "기대감": 0,
    "뿌듯함": 0,
    "편안/쾌적": 0,
    "신기함/관심": 0,
    "아껴주는": 0,
    "즐거움/신남": 0,
    "흐뭇함(귀여움/예쁨)": 0,
    "행복": 0,
    "기쁨": 0,
    "안심/신뢰": 0,

    # 1 분노/실망
    "불평/불만": 1,
    "화남/분노": 1,
    "안타까움/실망": 1,
    "의심/불신": 1,
    "한심함": 1,
    "역겨움/징그러움": 1,
    "짜증": 1,
    "어이없음": 1,
    "증오/혐오": 1,
    "경악": 1,

    # 2 체념/해탈
    "지긋지긋": 2,
    "슬픔": 2,
    "절망": 2,
    "패배/자기혐오": 2,
    "귀찮음": 2,
    "힘듦/지침": 2,
    "죄책감": 2,
    "당황/난처": 2,
    "부담/안_내킴": 2,
    "서러움": 2,
    "재미없음": 2,
    "불쌍함/연민": 2,
    "불안/걱정": 2,

    # 3 중립/정보성
    "없음": 3,
    "깨달음": 3,
    "놀람": 3,

    # 애매하지만 일단 분류
    "우쭐댐/무시함": 1,
    "비장함": 2,
    "부끄러움": 2,
    "공포/무서움": 2,
}

In [ ]:
id2label_4 = {
    0: "긍정/행복",
    1: "분노/실망",
    2: "체념/해탈",
    3: "중립/정보성",
}

def convert_kote_outputs_to_hanwha_4class(kote_outputs, threshold=0.25):
    """
    kote_outputs 예시:
    [
      {'label': '화남/분노', 'score': 0.72},
      {'label': '짜증', 'score': 0.61},
      ...
    ]
    """
    group_scores = {
        0: 0.0,
        1: 0.0,
        2: 0.0,
        3: 0.0,
    }

    used_reasons = []

    for item in kote_outputs:
        emotion = item["label"]
        score = item["score"]

        if score < threshold:
            continue

        if emotion in kote_to_hanwha_4:
            group = kote_to_hanwha_4[emotion]
            group_scores[group] += score
            used_reasons.append(f"{emotion}:{score:.3f}->{id2label_4[group]}")

    # 아무 감정도 threshold 이상 안 나오면 중립
    if max(group_scores.values()) == 0:
        return 3, "중립/정보성", 0.3, "no_emotion_over_threshold"

    pred_label = max(group_scores, key=group_scores.get)
    confidence = group_scores[pred_label] / (sum(group_scores.values()) + 1e-8)

    return pred_label, id2label_4[pred_label], confidence, "|".join(used_reasons)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
import pandas as pd
from tqdm import tqdm
import torch

model_name = "searle-j/kote_for_easygoing_people"

kote_tokenizer = AutoTokenizer.from_pretrained(model_name)
kote_model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1

kote_pipe = TextClassificationPipeline(
    model=kote_model,
    tokenizer=kote_tokenizer,
    device=device,
    top_k=None,
    function_to_apply="sigmoid"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
base_path = "/content/drive/MyDrive/Colab Notebooks"

df = pd.read_csv(f"{base_path}/hanwha_manual_labeling 1000.csv")

df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str)

In [ ]:
kote_pred_labels = []
kote_pred_names = []
kote_confidences = []
kote_reasons = []
kote_top5_raw = []

for text in tqdm(df["text"].tolist()):
    outputs = kote_pipe(text[:512])[0]

    # score 높은 순 정렬
    outputs = sorted(outputs, key=lambda x: x["score"], reverse=True)

    label, label_name, confidence, reason = convert_kote_outputs_to_hanwha_4class(
        outputs,
        threshold=0.25
    )

    kote_pred_labels.append(label)
    kote_pred_names.append(label_name)
    kote_confidences.append(confidence)
    kote_reasons.append(reason)
    kote_top5_raw.append(str(outputs[:5]))

df["kote_4class_label"] = kote_pred_labels
df["kote_4class_name"] = kote_pred_names
df["kote_4class_confidence"] = kote_confidences
df["kote_reason"] = kote_reasons
df["kote_top5_raw"] = kote_top5_raw

df[["text", "label", "label_name", "kote_4class_name", "kote_4class_confidence", "kote_top5_raw"]].head()

100%|██████████| 998/998 [00:09<00:00, 101.39it/s]


,text,label,label_name,kote_4class_name,kote_4class_confidence,kote_top5_raw
0,KBO 리그는 오직 티빙에서,3,NaN,긍정/행복,0.628560,"[{'label': '기대감', 'score': 0.8623221516609192}..."
1,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,3,NaN,분노/실망,0.726408,"[{'label': '안타까움/실망', 'score': 0.9178446531295..."
2,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네,0,NaN,긍정/행복,1.000000,"[{'label': '흐뭇함(귀여움/예쁨)', 'score': 0.955457925..."
3,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ,0,NaN,긍정/행복,0.816853,"[{'label': '즐거움/신남', 'score': 0.96100544929504..."
4,오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데,0,NaN,긍정/행복,0.882051,"[{'label': '감동/감탄', 'score': 0.776828110218048..."


In [ ]:
df.to_csv(
    f"{base_path}/hanwha_with_kote_4class_aux.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
compare_df = df[df["label"].isin([0, 1, 2, 3])].copy()

compare_df["label"] = compare_df["label"].astype(int)

agreement = (compare_df["label"] == compare_df["kote_4class_label"]).mean()

print("수동 라벨 vs KOTE 4분류 일치율:", agreement)

수동 라벨 vs KOTE 4분류 일치율: 0.6753507014028056


일치율은 0.67 3개 중 1는 맞는 수준이다.

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    compare_df["label"],
    compare_df["kote_4class_label"],
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=["수동_긍정", "수동_분노", "수동_체념", "수동_중립"],
    columns=["KOTE_긍정", "KOTE_분노", "KOTE_체념", "KOTE_중립"]
)

cm_df

,KOTE_긍정,KOTE_분노,KOTE_체념,KOTE_중립
수동_긍정,434,42,3,1
수동_분노,48,214,13,4
수동_체념,41,41,20,0
수동_중립,73,45,13,6


In [ ]:
mismatch_df = compare_df[
    compare_df["label"] != compare_df["kote_4class_label"]
].copy()

mismatch_df[
    ["text", "label_name", "kote_4class_name", "kote_4class_confidence", "kote_top5_raw"]
].head(50)

,text,label_name,kote_4class_name,kote_4class_confidence,kote_top5_raw
0,KBO 리그는 오직 티빙에서,NaN,긍정/행복,0.628560,"[{'label': '기대감', 'score': 0.8623221516609192}..."
1,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,NaN,분노/실망,0.726408,"[{'label': '안타까움/실망', 'score': 0.9178446531295..."
5,한동희 3경기 연속홈런이지만 그게 전부여서 아직 의심중인 롯데팬이면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋ,NaN,분노/실망,0.490900,"[{'label': '어이없음', 'score': 0.755485475063324}..."
9,유강남을 상대하던 팀들이 이런 기분이었을까,NaN,체념/해탈,0.440852,"[{'label': '안타까움/실망', 'score': 0.7294341921806..."
10,손성빈 수비력 진짜 대박이더라 개부럽다,NaN,긍정/행복,0.768961,"[{'label': '감동/감탄', 'score': 0.956837654113769..."
23,유강남 자동문 허인서 자동문 둘다 수비적으론 문제구만 ㅋ,NaN,분노/실망,0.786623,"[{'label': '어이없음', 'score': 0.8744617104530334..."
26,동희야 한겜당 홈런 하나씩만 치자,NaN,긍정/행복,1.000000,"[{'label': '기대감', 'score': 0.9077536463737488}..."
31,화요일 경기 슈퍼매치를 이기다니 ㄷㄷ,NaN,긍정/행복,0.670952,"[{'label': '놀람', 'score': 0.9203354120254517},..."
34,오늘 어케 이겼는지 감도 안잡히는 롯팬들 개추 ㅋㅋㅋ,NaN,분노/실망,0.577107,"[{'label': '어이없음', 'score': 0.882262110710144}..."
42,김경문의 대단한 용병술. 실망시키지않는 노시환,NaN,긍정/행복,0.867261,"[{'label': '감동/감탄', 'score': 0.975234270095825..."


In [ ]:
high_conf_mismatch = mismatch_df.sort_values(
    "kote_4class_confidence",
    ascending=False
)

high_conf_mismatch[
    ["text", "label_name", "kote_4class_name", "kote_4class_confidence", "kote_top5_raw"]
].head(100)

,text,label_name,kote_4class_name,kote_4class_confidence,kote_top5_raw
878,김서현은 사랑입니다,NaN,긍정/행복,1.000000,"[{'label': '아껴주는', 'score': 0.9362216591835022..."
888,엄상백 심우준 노시환 김서현 타팀 팬으로서 너무 사랑합니다~,NaN,긍정/행복,1.000000,"[{'label': '환영/호의', 'score': 0.932696402072906..."
672,역시 믿음이 가는 한화 입니다~~,NaN,긍정/행복,1.000000,"[{'label': '안심/신뢰', 'score': 0.945414304733276..."
647,한화본모습으로 돌아오니 너무반가운데 300억 짜리 2군행 정말좋은구단한화,NaN,긍정/행복,1.000000,"[{'label': '즐거움/신남', 'score': 0.95994520187377..."
847,사랑해요 김경문감독님 오늘도 승리 당했어요,NaN,긍정/행복,1.000000,"[{'label': '감동/감탄', 'score': 0.927541673183441..."
...,...,...,...,...,...
308,"한팬이지만 고영표 공 ㅈㄴ 맛도리 ,, 밸런스 계속 유지하는 게 신기하다",NaN,긍정/행복,0.806571,"[{'label': '감동/감탄', 'score': 0.954839110374450..."
176,기아가 최원준을 왜 크트로 보냈는지 불가사의,NaN,분노/실망,0.805431,"[{'label': '어이없음', 'score': 0.8784134984016418..."
912,이와중에 허인서 번트 지시는 진짜 개레전드였다,NaN,긍정/행복,0.803442,"[{'label': '감동/감탄', 'score': 0.964128494262695..."
480,이야... 안장군 등판일에 무슨일이고!,NaN,긍정/행복,0.802828,"[{'label': '놀람', 'score': 0.9250231981277466},..."


모델 사용해서 댓글 감정 분석

시드 고정

In [ ]:
import random
import numpy as np
import torch
from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


데이터 로드

In [ ]:
from datasets import load_dataset

base_path = "/content/drive/MyDrive/Colab Notebooks"

train_path = f"{base_path}/hanwha_train_4class_1000.csv"
valid_path = f"{base_path}/hanwha_valid_4class_1000.csv"
test_path = f"{base_path}/hanwha_test_4class_1000.csv"

dataset = load_dataset(
    "csv",
    data_files={
        "train": train_path,
        "validation": valid_path,
        "test": test_path,
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at'],
        num_rows: 694
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at'],
        num_rows: 149
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at'],
        num_rows: 149
    })
})

토크나이저 호출

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/694 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

In [ ]:
# label 컬럼만 labels로 바꾸기 전에 필요한 컬럼만 유지
keep_cols = ["input_ids", "attention_mask", "label"]

tokenized_dataset = tokenized_dataset.remove_columns(
    [
        col for col in tokenized_dataset["train"].column_names
        if col not in keep_cols
    ]
)

tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 694
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 149
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 149
    })
})

평가 함수

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],
        "macro_f1": f1.compute(
            predictions=preds,
            references=labels,
            average="macro"
        )["f1"],
        "weighted_f1": f1.compute(
            predictions=preds,
            references=labels,
            average="weighted"
        )["f1"],
    }

모델 학습

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir=f"{base_path}/hanwha_roberta_4class_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,

    seed=42,
    data_seed=42,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.110484,0.945992,0.644295,0.358346,0.555415
2,0.825019,0.857170,0.677852,0.386926,0.595311
3,0.527473,0.923362,0.651007,0.422363,0.611352
4,0.385563,0.881914,0.664430,0.473574,0.645295
5,0.437262,0.908871,0.651007,0.465932,0.640368


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=220, training_loss=0.7055199904875322, metrics={'train_runtime': 96.4481, 'train_samples_per_second': 35.978, 'train_steps_per_second': 2.281, 'total_flos': 228252939233280.0, 'train_loss': 0.7055199904875322, 'epoch': 5.0})

In [ ]:
baseline_result = trainer.evaluate(tokenized_dataset["test"])
baseline_result

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.437262,0.801555,5,0.697987,0.529775,0.679144


{'eval_loss': 0.8015550374984741,
 'eval_accuracy': 0.697986577181208,
 'eval_macro_f1': 0.5297748307701334,
 'eval_weighted_f1': 0.6791438054315888}

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

pred_output = trainer.predict(tokenized_dataset["test"])

preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

# Correct num_labels and id2label for the 4-class baseline evaluation
num_labels = 4
id2label = {
    0: "긍정/행복",
    1: "분노/실망",
    2: "체념/해탈",
    3: "중립/정보성",
}

label_names = [id2label[i] for i in range(num_labels)]

print(classification_report(
    labels,
    preds,
    target_names=label_names,
    digits=4
))

cm = confusion_matrix(labels, preds)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in label_names],
    columns=[f"pred_{name}" for name in label_names]
)

cm_df

              precision    recall  f1-score   support

       긍정/행복     0.7975    0.8750    0.8344        72
       분노/실망     0.7273    0.7619    0.7442        42
       체념/해탈     0.3333    0.1333    0.1905        15
      중립/정보성     0.3500    0.3500    0.3500        20

    accuracy                         0.6980       149
   macro avg     0.5520    0.5301    0.5298       149
weighted avg     0.6709    0.6980    0.6791       149



,pred_긍정/행복,pred_분노/실망,pred_체념/해탈,pred_중립/정보성
true_긍정/행복,63,2,2,5
true_분노/실망,4,32,1,5
true_체념/해탈,3,7,2,3
true_중립/정보성,9,3,1,7


기쁨/행복 분노/실망 라벨은 데이터 수가 많아 비교적 잘 예측되었다.   
반면 체념/해탈과 중립/정보성은 데이터 수가 적고 표현이 유사해 오분류가 발생하였다.

In [ ]:
result_row = {
    "experiment": "Exp0_roberta_4class_baseline",
    "model": "klue/roberta-base",
    "method": "basic_finetuning",
    "accuracy": baseline_result["eval_accuracy"],
    "macro_f1": baseline_result["eval_macro_f1"],
    "weighted_f1": baseline_result["eval_weighted_f1"],
}

result_df = pd.DataFrame([result_row])

result_df.to_csv(
    f"{base_path}/hanwha_experiment_results.csv",
    index=False,
    encoding="utf-8-sig"
)

result_df

,experiment,model,method,accuracy,macro_f1,weighted_f1
0,Exp0_roberta_4class_baseline,klue/roberta-base,basic_finetuning,0.697987,0.529775,0.679144


In [ ]:
save_path = f"{base_path}/hanwha_roberta_4class_best"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("저장 완료:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/Colab Notebooks/hanwha_roberta_4class_best


In [ ]:
from transformers import pipeline

sentiment_pipe = pipeline(
    "text-classification",
    model=save_path,
    tokenizer=save_path,
    device=0 if torch.cuda.is_available() else -1
)

test_comments = [
    "나는 행복합니다 ㅋㅋㅋㅋ 또 졌네",
    "오늘 한화 진짜 잘했다",
    "불펜 진짜 답없다",
    "오늘 선발 누구임?",
    "역시 한화답다 이젠 화도 안 난다",
]

for comment in test_comments:
    print(comment)
    print(sentiment_pipe(comment))
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

나는 행복합니다 ㅋㅋㅋㅋ 또 졌네
[{'label': '기쁨/행복', 'score': 0.907160758972168}]

오늘 한화 진짜 잘했다
[{'label': '기쁨/행복', 'score': 0.956788957118988}]

불펜 진짜 답없다
[{'label': '분노/실망', 'score': 0.8094000816345215}]

오늘 선발 누구임?
[{'label': '기쁨/행복', 'score': 0.816375195980072}]

역시 한화답다 이젠 화도 안 난다
[{'label': '분노/실망', 'score': 0.7064920663833618}]



현대 LLM에 비해 작은 fineturning 모델이 반어법이나 인터넷밈 표현을 잘 파악못하는 이유는 문맥 전체를 보는게 아니라 토큰을 각각 살펴서 이다.  
문맥적인 정보를 보조적으로 떠먹여 줘야 LLM스럽게 사고 할수 있지 않을까 한다.

이후 실험 과정은    
Exp1 Class Weight	체념/중립 recall 개선   
Exp2 KOTE Aux Input	작은 모델에 감정 힌트 주입    
Exp3 KcELECTRA	댓글체 모델 비교    

In [ ]:
results = []

results.append({
    "experiment": "klue_roberta_baseline",
    "model": "klue/roberta-base",
    "method": "basic_finetuning",
    **baseline_result
})

result_df = pd.DataFrame(results)
result_df

,experiment,model,method,eval_loss,eval_accuracy,eval_macro_f1,eval_weighted_f1
0,klue_roberta_baseline,klue/roberta-base,basic_finetuning,0.904933,0.68,0.344486,0.598791


클래스 불균형 문제를 해결하기 위해 class weight를 적용해보기

In [ ]:
import numpy as np
import torch
import pandas as pd

from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EarlyStoppingCallback
from torch import nn

train_df = pd.read_csv(train_path)

train_labels = train_df["label"].values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("class weights:", class_weights)

for i, w in enumerate(class_weights):
    print(id2label[i], float(w))

class weights: tensor([0.5164, 0.8897, 2.4437, 1.8859])
긍정/행복 0.5163690447807312
분노/실망 0.8897435665130615
체념/해탈 2.44366192817688
중립/정보성 1.8858696222305298


웨이트 트레이너 정의

In [ ]:
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = self.class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

모델 로드

In [ ]:
weighted_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


트레이닝 어그먼트

In [ ]:
weighted_args = TrainingArguments(
    output_dir=f"{base_path}/hanwha_roberta_4class_class_weight",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,

    seed=42,
    data_seed=42,
)

모델 학습

In [ ]:
weighted_trainer = WeightedLossTrainer(
    model=weighted_model,
    args=weighted_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

weighted_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.411085,1.349529,0.416107,0.278914,0.400473
2,1.115150,1.091879,0.624161,0.483051,0.631443
3,0.837305,1.040413,0.644295,0.515275,0.658552
4,0.600678,1.139309,0.624161,0.491958,0.634752
5,0.550173,1.157581,0.624161,0.509182,0.639270


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=220, training_loss=0.9593872742219405, metrics={'train_runtime': 120.5874, 'train_samples_per_second': 28.776, 'train_steps_per_second': 1.824, 'total_flos': 228252939233280.0, 'train_loss': 0.9593872742219405, 'epoch': 5.0})

In [ ]:
class_weight_result = weighted_trainer.evaluate(tokenized_dataset["test"])
class_weight_result

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.550173,1.013030,5,0.644295,0.504999,0.655563


{'eval_loss': 1.0130295753479004,
 'eval_accuracy': 0.6442953020134228,
 'eval_macro_f1': 0.5049988101179512,
 'eval_weighted_f1': 0.6555625745353546}

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_output = weighted_trainer.predict(tokenized_dataset["test"])

preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

label_names = [id2label[i] for i in range(num_labels)]

print(classification_report(
    labels,
    preds,
    target_names=label_names,
    digits=4
))

cm = confusion_matrix(labels, preds, labels=list(range(num_labels)))

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in label_names],
    columns=[f"pred_{name}" for name in label_names]
)

cm_df

              precision    recall  f1-score   support

       긍정/행복     0.8710    0.7500    0.8060        72
       분노/실망     0.7436    0.6905    0.7160        42
       체념/해탈     0.0909    0.0667    0.0769        15
      중립/정보성     0.3243    0.6000    0.4211        20

    accuracy                         0.6443       149
   macro avg     0.5074    0.5268    0.5050       149
weighted avg     0.6832    0.6443    0.6556       149



,pred_긍정/행복,pred_분노/실망,pred_체념/해탈,pred_중립/정보성
true_긍정/행복,54,0,5,13
true_분노/실망,1,29,3,9
true_체념/해탈,3,8,1,3
true_중립/정보성,4,2,2,12


채념/해탈 라벨 개선 실패했고 중립/ 정보성 라벨의 f1 스코어는 상당히 많이 개선 되었다.

In [ ]:
result_row = {
    "experiment": "Exp1_roberta_4class_class_weight",
    "model": BASE_MODEL,
    "method": "class_weight",
    "accuracy": class_weight_result["eval_accuracy"],
    "macro_f1": class_weight_result["eval_macro_f1"],
    "weighted_f1": class_weight_result["eval_weighted_f1"],
}

result_path = f"{base_path}/hanwha_experiment_results.csv"

try:
    result_df = pd.read_csv(result_path)
    result_df = pd.concat([result_df, pd.DataFrame([result_row])], ignore_index=True)
except FileNotFoundError:
    result_df = pd.DataFrame([result_row])

result_df.to_csv(
    result_path,
    index=False,
    encoding="utf-8-sig"
)

result_df

,experiment,model,method,accuracy,macro_f1,weighted_f1
0,Exp0_roberta_4class_baseline,klue/roberta-base,basic_finetuning,0.697987,0.529775,0.679144
1,Exp1_roberta_4class_class_weight,klue/roberta-base,class_weight,0.644295,0.504999,0.655563


Exp2: KOTE Aux Input

보조 데이터 만들기

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

base_path = "/content/drive/MyDrive/Colab Notebooks"

kote_aux_path = f"{base_path}/hanwha_with_kote_4class_aux.csv"

df = pd.read_csv(kote_aux_path)

df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label", "text"]).copy()
df["label"] = df["label"].astype(int)

# 4분류만 사용
df = df[df["label"].isin([0, 1, 2, 3])].copy()

id2label = {
    0: "긍정/행복",
    1: "분노/실망",
    2: "체념/해탈",
    3: "중립/정보성",
}

df["label_name"] = df["label"].map(id2label)

print(len(df))
print(df["label_name"].value_counts())

998
label_name
긍정/행복     480
분노/실망     279
중립/정보성    137
체념/해탈     102
Name: count, dtype: int64


In [ ]:
def make_kote_aux_input(row):
    text = str(row["text"]).strip()

    if "kote_reason" in row and pd.notna(row["kote_reason"]):
        kote_info = str(row["kote_reason"])
    elif "kote_top5_raw" in row and pd.notna(row["kote_top5_raw"]):
        kote_info = str(row["kote_top5_raw"])
    else:
        kote_info = ""

    # 너무 길면 잘라서 노이즈 방지
    kote_info = kote_info[:180]

    return f"{text} [SEP] 보조감정: {kote_info}"

In [ ]:
df["model_input"] = df.apply(make_kote_aux_input, axis=1)

df[["text", "model_input", "label_name"]].head()

,text,model_input,label_name
0,KBO 리그는 오직 티빙에서,KBO 리그는 오직 티빙에서 [SEP] 보조감정: 기대감:0.862->긍정/행복|없...,중립/정보성
1,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추,한동희 이제 살아나나싶지만 당한게 많아 아직도 의심하는 롯데팬이면 개추 [SEP] ...,중립/정보성
2,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네,손성빈 진짜 좋다ㅋㅋㅋ 멀티히트에 호수비에ㅋㅋ 오늘 골고루 활약해서 더 보기가좋네 ...,긍정/행복
3,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ,손성빈이 롯데 주전포수면 개추 ㅋㅋㅋㅋㅋㅋㅋㅋㅋ [SEP] 보조감정: 즐거움/신남:...,긍정/행복
4,오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데,오늘 경기 전민재 칭찬이 많이 없네 홈런에 호수비에 활약 좋았는데 [SEP] 보조감...,긍정/행복


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

print("\nTrain")
print(train_df["label_name"].value_counts())

print("\nValid")
print(valid_df["label_name"].value_counts())

print("\nTest")
print(test_df["label_name"].value_counts())

train: 698
valid: 150
test: 150

Train
label_name
긍정/행복     336
분노/실망     195
중립/정보성     96
체념/해탈      71
Name: count, dtype: int64

Valid
label_name
긍정/행복     72
분노/실망     42
중립/정보성    21
체념/해탈     15
Name: count, dtype: int64

Test
label_name
긍정/행복     72
분노/실망     42
중립/정보성    20
체념/해탈     16
Name: count, dtype: int64


In [ ]:
train_aux_path = f"{base_path}/hanwha_train_kote_aux.csv"
valid_aux_path = f"{base_path}/hanwha_valid_kote_aux.csv"
test_aux_path = f"{base_path}/hanwha_test_kote_aux.csv"

train_df.to_csv(train_aux_path, index=False, encoding="utf-8-sig")
valid_df.to_csv(valid_aux_path, index=False, encoding="utf-8-sig")
test_df.to_csv(test_aux_path, index=False, encoding="utf-8-sig")

데이터 셋 로드

In [ ]:
from datasets import load_dataset

dataset_aux = load_dataset(
    "csv",
    data_files={
        "train": train_aux_path,
        "validation": valid_aux_path,
        "test": test_aux_path,
    }
)

dataset_aux

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'kote_4class_label', 'kote_4class_name', 'kote_4class_confidence', 'kote_reason', 'kote_top5_raw', 'model_input'],
        num_rows: 698
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'kote_4class_label', 'kote_4class_name', 'kote_4class_confidence', 'kote_reason', 'kote_top5_raw', 'model_input'],
        num_rows: 150
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'kote_4class_label', 'kote_4class_name', 'kote_4class_confidence', 'kote_reason', 'kote_top5_raw', 'model_input'],


토크나이징

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_aux_fn(batch):
    return tokenizer(
        batch["model_input"],
        truncation=True,
        padding="max_length",
        max_length=192,
    )

tokenized_aux_dataset = dataset_aux.map(tokenize_aux_fn, batched=True)

Map:   0%|          | 0/698 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
keep_cols = ["input_ids", "attention_mask", "label"]

tokenized_aux_dataset = tokenized_aux_dataset.remove_columns(
    [
        col for col in tokenized_aux_dataset["train"].column_names
        if col not in keep_cols
    ]
)

tokenized_aux_dataset = tokenized_aux_dataset.rename_column("label", "labels")

tokenized_aux_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_aux_dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 698
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 150
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 150
    })
})

모델 학습

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

num_labels = 4

id2label = {
    0: "긍정/행복",
    1: "분노/실망",
    2: "체념/해탈",
    3: "중립/정보성",
}

label2id = {v: k for k, v in id2label.items()}

aux_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
aux_args = TrainingArguments(
    output_dir=f"{base_path}/hanwha_roberta_4class_kote_aux",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,

    seed=42,
    data_seed=42,
)

In [ ]:
aux_trainer = Trainer(
    model=aux_model,
    args=aux_args,
    train_dataset=tokenized_aux_dataset["train"],
    eval_dataset=tokenized_aux_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

aux_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.003651,0.891905,0.686667,0.470815,0.646786
2,0.803669,0.820163,0.693333,0.502906,0.665279
3,0.644979,0.772058,0.713333,0.509237,0.676977
4,0.522369,0.796453,0.713333,0.502033,0.670569
5,0.424222,0.781305,0.726667,0.530214,0.691489


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=220, training_loss=0.6585115692832253, metrics={'train_runtime': 107.1808, 'train_samples_per_second': 32.562, 'train_steps_per_second': 2.053, 'total_flos': 344352777200640.0, 'train_loss': 0.6585115692832253, 'epoch': 5.0})

테스트셋 평가

In [ ]:
kote_aux_result = aux_trainer.evaluate(tokenized_aux_dataset["test"])
kote_aux_result

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.424222,0.766112,5,0.706667,0.549524,0.692665


{'eval_loss': 0.7661116123199463,
 'eval_accuracy': 0.7066666666666667,
 'eval_macro_f1': 0.5495242454310165,
 'eval_weighted_f1': 0.6926648470998585}

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

pred_output = aux_trainer.predict(tokenized_aux_dataset["test"])

preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

label_names = [id2label[i] for i in range(num_labels)]

print(classification_report(
    labels,
    preds,
    target_names=label_names,
    digits=4
))

cm = confusion_matrix(labels, preds, labels=list(range(num_labels)))

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in label_names],
    columns=[f"pred_{name}" for name in label_names]
)

cm_df

              precision    recall  f1-score   support

       긍정/행복     0.8182    0.8750    0.8456        72
       분노/실망     0.7556    0.8095    0.7816        42
       체념/해탈     0.4000    0.2500    0.3077        16
      중립/정보성     0.2778    0.2500    0.2632        20

    accuracy                         0.7067       150
   macro avg     0.5629    0.5461    0.5495       150
weighted avg     0.6840    0.7067    0.6927       150



,pred_긍정/행복,pred_분노/실망,pred_체념/해탈,pred_중립/정보성
true_긍정/행복,63,1,3,5
true_분노/실망,3,34,0,5
true_체념/해탈,6,3,4,3
true_중립/정보성,5,7,3,5


5에포크 더 학습시 체념/해탈 f1점수는 0.25로 올라갔지만 중립/정보 f1 점수는 0.6에서 0.25로 트레이드 오프 되었다.

In [ ]:
result_row = {
    "experiment": "Exp2_roberta_4class_kote_aux_input",
    "model": BASE_MODEL,
    "method": "kote_aux_input",
    "accuracy": kote_aux_result["eval_accuracy"],
    "macro_f1": kote_aux_result["eval_macro_f1"],
    "weighted_f1": kote_aux_result["eval_weighted_f1"],
}

result_path = f"{base_path}/hanwha_experiment_results.csv"

try:
    result_df = pd.read_csv(result_path)
    result_df = pd.concat([result_df, pd.DataFrame([result_row])], ignore_index=True)
except FileNotFoundError:
    result_df = pd.DataFrame([result_row])

result_df.to_csv(
    result_path,
    index=False,
    encoding="utf-8-sig"
)

result_df

,experiment,model,method,accuracy,macro_f1,weighted_f1
0,Exp0_roberta_4class_baseline,klue/roberta-base,basic_finetuning,0.697987,0.529775,0.679144
1,Exp1_roberta_4class_class_weight,klue/roberta-base,class_weight,0.644295,0.504999,0.655563
2,Exp2_roberta_4class_kote_aux_input,klue/roberta-base,kote_aux_input,0.686667,0.463016,0.630503


KOTE 기반 보조 감정 정보를 댓글 입력에 추가하는 실험을 수행했으나, 기본 RoBERTa 모델 대비 Macro F1이 하락하였다. 특히 체념/해탈 클래스는 F1-score가 0으로 나타났다. 이는 KOTE의 세부 감정 라벨이 한화 팬덤 특유의 반어적·자조적 표현을 직접적으로 설명하지 못하고, 오히려 분노/실망이나 긍정/행복 계열 정보로 모델 입력을 혼란스럽게 만들었기 때문으로 해석된다. 따라서 KOTE는 모델 입력으로 직접 사용하는 것보다, 수동 라벨 검수나 오류 분석 보조 도구로 사용하는 것이 더 적합하다.

기존 실험 설계를 재정의하여 긍정/부정 이진 감정분류 모델로 실험 결과 안정성을 올리고 한화 댓글속 행복이 진짜 행복인지 다시금 알아보고자 한다.

NSMR 데이터로 1차 fineturning

In [ ]:
!wget -O /content/ratings_train.txt https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
!wget -O /content/ratings_test.txt https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt

--2026-05-22 03:09:05--  https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14628807 (14M) [text/plain]
Saving to: ‘/content/ratings_train.txt’

/content/ratings_tr 100%[===================>]  13.95M  --.-KB/s    in 0.08s   

2026-05-22 03:09:06 (169 MB/s) - ‘/content/ratings_train.txt’ saved [14628807/14628807]

--2026-05-22 03:09:06--  https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4893335 (4.7M) [

In [ ]:
import pandas as pd

nsmc_train_df = pd.read_csv(
    "/content/ratings_train.txt",
    sep="\t"
)

nsmc_test_df = pd.read_csv(
    "/content/ratings_test.txt",
    sep="\t"
)

nsmc_train_df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [ ]:
nsmc_train_df = nsmc_train_df.dropna(subset=["document", "label"])
nsmc_test_df = nsmc_test_df.dropna(subset=["document", "label"])

nsmc_train_df["document"] = nsmc_train_df["document"].astype(str)
nsmc_test_df["document"] = nsmc_test_df["document"].astype(str)

print(nsmc_train_df.shape)
print(nsmc_test_df.shape)
print(nsmc_train_df["label"].value_counts())

(149995, 3)
(49997, 3)
label
0    75170
1    74825
Name: count, dtype: int64


In [ ]:
from datasets import Dataset, DatasetDict

# 빠른 실험용으로 일부만 사용
nsmc_train_sample = nsmc_train_df.sample(
    n=30000,
    random_state=42
)

nsmc_valid_sample = nsmc_test_df.sample(
    n=5000,
    random_state=42
)

nsmc_dataset = DatasetDict({
    "train": Dataset.from_pandas(nsmc_train_sample.reset_index(drop=True)),
    "validation": Dataset.from_pandas(nsmc_valid_sample.reset_index(drop=True)),
})

nsmc_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 5000
    })
})

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_nsmc(batch):
    return tokenizer(
        batch["document"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

nsmc_tok = nsmc_dataset.map(tokenize_nsmc, batched=True)

nsmc_tok = nsmc_tok.remove_columns(
    [
        col for col in nsmc_tok["train"].column_names
        if col not in ["input_ids", "attention_mask", "label"]
    ]
)

nsmc_tok = nsmc_tok.rename_column("label", "labels")

nsmc_tok.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

nsmc_tok

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
})

In [ ]:
import torch
import evaluate
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics_binary(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=labels
        )["f1"],
    }

model_nsmc = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

base_path = "/content/drive/MyDrive/Colab Notebooks"

nsmc_args = TrainingArguments(
    output_dir=f"{base_path}/roberta_nsmc_stage1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
)

nsmc_trainer = Trainer(
    model=model_nsmc,
    args=nsmc_args,
    train_dataset=nsmc_tok["train"],
    eval_dataset=nsmc_tok["validation"],
    compute_metrics=compute_metrics_binary,
)

nsmc_trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
nsmc_trainer.save_model("/content/drive/MyDrive/Colab Notebooks/roberta_nsmc_stage1_best")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/roberta_nsmc_stage1_best")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/roberta_nsmc_stage1_best/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/roberta_nsmc_stage1_best/tokenizer.json')

In [ ]:
nsmc_result = nsmc_trainer.evaluate(nsmc_tok["validation"])
nsmc_result

Epoch,Training Loss,Validation Loss,Accuracy,F1
0,0.373940,0.337924,0.857200,0.854464


{'eval_loss': 0.33792445063591003,
 'eval_accuracy': 0.8572,
 'eval_f1': 0.8544639217284957}

4분류 데이터 2분류로 변환

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

base_path = "/content/drive/MyDrive/Colab Notebooks"

# 현재 수동 라벨링 파일 경로
data_path = f"{base_path}/hanwha_manual_labeling 1000.csv"

df = pd.read_csv(data_path)

# label 정리
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label", "text"]).copy()
df["label"] = df["label"].astype(int)
df["text"] = df["text"].astype(str)

# 4분류/제외 → 이진 분류
def map_to_binary(label):
    if label == 0:          # 긍정/행복
        return 1            # 긍정
    elif label in [1, 2]:    # 분노/실망, 체념/해탈
        return 0            # 부정
    else:
        return -1           # 중립/정보성, 제외

df["binary_label"] = df["label"].apply(map_to_binary)

# 긍정/부정만 사용
binary_df = df[df["binary_label"].isin([0, 1])].copy()

binary_df["label"] = binary_df["binary_label"]

id2label_binary = {
    0: "부정",
    1: "긍정",
}

binary_df["label_name"] = binary_df["label"].map(id2label_binary)

# 중복 제거
binary_df = binary_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("이진 분류 데이터 수:", len(binary_df))
print(binary_df["label_name"].value_counts())

이진 분류 데이터 수: 860
label_name
긍정    480
부정    380
Name: count, dtype: int64


In [ ]:
train_df, temp_df = train_test_split(
    binary_df,
    test_size=0.3,
    random_state=42,
    stratify=binary_df["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

print("\nTrain")
print(train_df["label_name"].value_counts())

print("\nValid")
print(valid_df["label_name"].value_counts())

print("\nTest")
print(test_df["label_name"].value_counts())

train: 602
valid: 129
test: 129

Train
label_name
긍정    336
부정    266
Name: count, dtype: int64

Valid
label_name
긍정    72
부정    57
Name: count, dtype: int64

Test
label_name
긍정    72
부정    57
Name: count, dtype: int64


In [ ]:
train_path = f"{base_path}/hanwha_train_binary.csv"
valid_path = f"{base_path}/hanwha_valid_binary.csv"
test_path = f"{base_path}/hanwha_test_binary.csv"

train_df.to_csv(train_path, index=False, encoding="utf-8-sig")
valid_df.to_csv(valid_path, index=False, encoding="utf-8-sig")
test_df.to_csv(test_path, index=False, encoding="utf-8-sig")

print("저장 완료")
print(train_path)
print(valid_path)
print(test_path)

저장 완료
/content/drive/MyDrive/Colab Notebooks/hanwha_train_binary.csv
/content/drive/MyDrive/Colab Notebooks/hanwha_valid_binary.csv
/content/drive/MyDrive/Colab Notebooks/hanwha_test_binary.csv


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files={
        "train": train_path,
        "validation": valid_path,
        "test": test_path,
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'binary_label'],
        num_rows: 602
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'binary_label'],
        num_rows: 129
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'binary_label'],
        num_rows: 129
    })
})

In [ ]:
num_labels = 2

id2label = {
    0: "부정",
    1: "긍정",
}

label2id = {
    "부정": 0,
    "긍정": 1,
}

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_fn, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]

tokenized_dataset = tokenized_dataset.remove_columns(
    [
        col for col in tokenized_dataset["train"].column_names
        if col not in keep_cols
    ]
)

tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_dataset

Map:   0%|          | 0/602 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 602
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 129
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 129
    })
})

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=labels,
            average="binary",
            pos_label=1
        )["f1"],
        "macro_f1": f1.compute(
            predictions=preds,
            references=labels,
            average="macro"
        )["f1"],
    }

2분류 한화 댓글 fineturning

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir=f"{base_path}/hanwha_roberta_binary",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,

    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Macro F1
1,0.584342,0.324979,0.875969,0.893333,0.872593
2,0.405808,0.266300,0.906977,0.914286,0.906295
3,0.209028,0.232484,0.937984,0.945205,0.936888
4,0.128474,0.261021,0.922481,0.930556,0.921418
5,0.134853,0.274650,0.922481,0.930556,0.921418


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=190, training_loss=0.27674355867661926, metrics={'train_runtime': 122.6175, 'train_samples_per_second': 24.548, 'train_steps_per_second': 1.55, 'total_flos': 197991069158400.0, 'train_loss': 0.27674355867661926, 'epoch': 5.0})

테스트셋으로 평가

In [ ]:
binary_result = trainer.evaluate(tokenized_dataset["test"])
binary_result

Training Loss,Validation Loss,Epoch,Accuracy,F1,Macro F1
0.134853,0.183263,5,0.945736,0.953020,0.944400


{'eval_loss': 0.18326301872730255,
 'eval_accuracy': 0.9457364341085271,
 'eval_f1': 0.9530201342281879,
 'eval_macro_f1': 0.9443999753709746}

분류 리포트

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

pred_output = trainer.predict(tokenized_dataset["test"])

preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

label_names = [id2label[i] for i in range(num_labels)]

print(classification_report(
    labels,
    preds,
    target_names=label_names,
    digits=4
))

cm = confusion_matrix(labels, preds, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["true_부정", "true_긍정"],
    columns=["pred_부정", "pred_긍정"]
)

cm_df

              precision    recall  f1-score   support

          부정     0.9808    0.8947    0.9358        57
          긍정     0.9221    0.9861    0.9530        72

    accuracy                         0.9457       129
   macro avg     0.9514    0.9404    0.9444       129
weighted avg     0.9480    0.9457    0.9454       129



,pred_부정,pred_긍정
true_부정,51,6
true_긍정,1,71


In [ ]:
result_row = {
    "experiment": "Exp_binary_roberta_baseline",
    "model": BASE_MODEL,
    "method": "binary_finetuning",
    "accuracy": binary_result["eval_accuracy"],
    "f1": binary_result["eval_f1"],
    "macro_f1": binary_result["eval_macro_f1"],
}

result_path = f"{base_path}/hanwha_experiment_results_binary.csv"

result_df = pd.DataFrame([result_row])

result_df.to_csv(
    result_path,
    index=False,
    encoding="utf-8-sig"
)

result_df

,experiment,model,method,accuracy,f1,macro_f1
0,Exp_binary_roberta_baseline,klue/roberta-base,binary_finetuning,0.945736,0.95302,0.9444


In [ ]:
save_path = f"{base_path}/hanwha_roberta_binary_best"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("저장 완료:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/Colab Notebooks/hanwha_roberta_binary_best


NSMR 데이터로 1차 fineturing한 모델을 불러와서 한화댓글 데이터로 2차 fineturning

In [ ]:
stage1_model_path = "/content/drive/MyDrive/Colab Notebooks/roberta_nsmc_stage1_best"

In [ ]:
from datasets import load_dataset

base_path = "/content/drive/MyDrive/Colab Notebooks"

train_path = f"{base_path}/hanwha_train_binary.csv"
valid_path = f"{base_path}/hanwha_valid_binary.csv"
test_path = f"{base_path}/hanwha_test_binary.csv"

dataset = load_dataset(
    "csv",
    data_files={
        "train": train_path,
        "validation": valid_path,
        "test": test_path,
    }
)

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'binary_label'],
        num_rows: 602
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'binary_label'],
        num_rows: 129
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'memo', 'video_title', 'video_id', 'video_url', 'game_date', 'opponent', 'result', 'score', 'like_count', 'published_at', 'binary_label'],
        num_rows: 129
    })
})

In [ ]:
num_labels = 2

id2label = {
    0: "부정",
    1: "긍정",
}

label2id = {
    "부정": 0,
    "긍정": 1,
}

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(stage1_model_path)

In [ ]:
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_fn, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]

tokenized_dataset = tokenized_dataset.remove_columns(
    [
        col for col in tokenized_dataset["train"].column_names
        if col not in keep_cols
    ]
)

tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_dataset

Map:   0%|          | 0/602 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 602
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 129
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 129
    })
})

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=labels,
            average="binary",
            pos_label=1
        )["f1"],
        "macro_f1": f1.compute(
            predictions=preds,
            references=labels,
            average="macro"
        )["f1"],
    }

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    stage1_model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
import torch
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=f"{base_path}/roberta_nsmc_to_hanwha_binary",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,

    seed=42,
    data_seed=42,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Macro F1
1,0.395365,0.332960,0.868217,0.885906,0.864971
2,0.354963,0.289190,0.883721,0.897959,0.881412
3,0.213342,0.284634,0.906977,0.916667,0.905702
4,0.139386,0.338941,0.906977,0.917808,0.905333
5,0.159081,0.341601,0.914729,0.924138,0.913396


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=190, training_loss=0.23360160965668528, metrics={'train_runtime': 125.3591, 'train_samples_per_second': 24.011, 'train_steps_per_second': 1.516, 'total_flos': 197991069158400.0, 'train_loss': 0.23360160965668528, 'epoch': 5.0})

테스트 평가

In [ ]:
stage2_result = trainer.evaluate(tokenized_dataset["test"])
stage2_result

Training Loss,Validation Loss,Epoch,Accuracy,F1,Macro F1
0.159081,0.258265,5,0.930233,0.939597,0.928514


{'eval_loss': 0.25826531648635864,
 'eval_accuracy': 0.9302325581395349,
 'eval_f1': 0.9395973154362416,
 'eval_macro_f1': 0.928514254048396}

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

pred_output = trainer.predict(tokenized_dataset["test"])

preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

label_names = [id2label[i] for i in range(num_labels)]

print(classification_report(
    labels,
    preds,
    target_names=label_names,
    digits=4
))

cm = confusion_matrix(labels, preds, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["true_부정", "true_긍정"],
    columns=["pred_부정", "pred_긍정"]
)

cm_df

              precision    recall  f1-score   support

          부정     0.9615    0.8772    0.9174        57
          긍정     0.9091    0.9722    0.9396        72

    accuracy                         0.9302       129
   macro avg     0.9353    0.9247    0.9285       129
weighted avg     0.9323    0.9302    0.9298       129



,pred_부정,pred_긍정
true_부정,50,7
true_긍정,2,70


In [ ]:
result_row = {
    "experiment": "Exp_nsmc_stage1_to_hanwha_binary",
    "model": "klue/roberta-base",
    "method": "NSMC_1st_finetuning_then_Hanwha_2nd_finetuning",
    "accuracy": stage2_result["eval_accuracy"],
    "f1": stage2_result["eval_f1"],
    "macro_f1": stage2_result["eval_macro_f1"],
}

result_path = f"{base_path}/hanwha_experiment_results_binary.csv"

try:
    result_df = pd.read_csv(result_path)
    result_df = pd.concat([result_df, pd.DataFrame([result_row])], ignore_index=True)
except FileNotFoundError:
    result_df = pd.DataFrame([result_row])

result_df.to_csv(
    result_path,
    index=False,
    encoding="utf-8-sig"
)

result_df

,experiment,model,method,accuracy,f1,macro_f1
0,Exp_binary_roberta_baseline,klue/roberta-base,binary_finetuning,0.945736,0.953020,0.944400
1,Exp_nsmc_stage1_to_hanwha_binary,klue/roberta-base,NSMC_1st_finetuning_then_Hanwha_2nd_finetuning,0.930233,0.939597,0.928514


In [ ]:
save_path = f"{base_path}/roberta_nsmc_to_hanwha_binary_best"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("저장 완료:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/Colab Notebooks/roberta_nsmc_to_hanwha_binary_best


In [ ]:
from transformers import pipeline

sentiment_pipe = pipeline(
    "text-classification",
    model=save_path,
    tokenizer=save_path,
    device=0 if torch.cuda.is_available() else -1
)

test_comments = [
    "오늘 한화 진짜 잘했다",
    "불펜 진짜 답없다",
    "나는 행복합니다 ㅋㅋㅋㅋ 또 졌네",
    "문동주 오늘 미쳤다",
    "역시 한화답다 이젠 화도 안 난다",
]

for comment in test_comments:
    print(comment)
    print(sentiment_pipe(comment))
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

오늘 한화 진짜 잘했다
[{'label': '긍정', 'score': 0.9932952523231506}]

불펜 진짜 답없다
[{'label': '부정', 'score': 0.9917245507240295}]

나는 행복합니다 ㅋㅋㅋㅋ 또 졌네
[{'label': '부정', 'score': 0.696692943572998}]

문동주 오늘 미쳤다
[{'label': '긍정', 'score': 0.9920729994773865}]

역시 한화답다 이젠 화도 안 난다
[{'label': '긍정', 'score': 0.6515576243400574}]



비꼬거나 반어적 표현에 대해서 구별하지 못한다.   
개선 방법 MEME TOKEN을 사용해서 문맥상 반어적 표현을 알려주기

In [ ]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/happy.ipynb"

with open/notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# metadata.widgets 제거
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

fixed_path = notebook_path.replace(".ipynb", "_fixed.ipynb")

with open(fixed_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("수정된 파일 저장:", fixed_path)

SyntaxError: unmatched ')' (3937711192.py, line 5)